In [2]:
!pip install pdfplumber -q

In [ ]:
import pdfplumber, re, json
from pathlib import Path

PDF_PATH = "/Users/gimseoyeon/Downloads/AICPP/.venv/인공지능기본법_지원데스크_사례집.pdf"   
OUTPUT_PATH = "사례집_chunks.jsonl"


## 전체 텍스트 추출

In [7]:
def extract_pages(pdf_path):
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            t = page.extract_text()
            if t and t.strip():
                pages.append({"page": i+1, "text": t.strip()})
    print(f"유효 페이지: {len(pages)}개")
    return pages

pages = extract_pages(PDF_PATH)
print(pages[0]["text"][:300])


유효 페이지: 125개
인공지능기본법 지원데스크 사례집
20선의 질답 사례로 배우는 기업 실무 가이드


## 클리닝

In [8]:
def clean(text):
    text = re.sub(r'인공지능기본법 지원데스크 사례집\s*', '', text)
    text = re.sub(r'\[20선의 질답 사례로 배우는 기업 실무 가이드\]\s*', '', text)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    text = text.replace('Ÿ', '•').replace('ㅅ', '')
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)
    return text.strip()

for p in pages:
    p["text_clean"] = clean(p["text"])

full = "\n\n".join(p["text_clean"] for p in pages)
print(f"전체 {len(full)}자")


전체 128613자


## 파트 분리

In [9]:
def find_pos(pattern, text):
    m = re.search(pattern, text)
    return m.start() if m else None

p1 = find_pos(r'제1장', full)
p2 = find_pos(r'제2장', full)
p3 = find_pos(r'제3장', full)

print(f"제1장 위치: {p1}, 제2장: {p2}, 제3장: {p3}")

part2_text = full[p2:p3] if p2 and p3 else ""
part3_text = full[p3:] if p3 else full


제1장 위치: 63, 제2장: 153, 제3장: 343


##  Q&A 사례 추출

In [10]:
def extract_qa_cases(text):
    chunks = []

    # 유형 위치 파악
    type_map = {}
    for m in re.finditer(r'유형\s*(\d+)\s+([^\n]+)', text):
        type_map[m.start()] = {"유형번호": m.group(1), "유형명": m.group(2).strip()}

    def get_type(pos):
        for s in sorted(type_map.keys(), reverse=True):
            if s <= pos:
                return type_map[s]
        return {"유형번호": "0", "유형명": "미분류"}

    # 사례 분리
    case_pat = re.compile(
        r'<사례\s*(\d+)>\s*([^\n]+)\n(.*?)(?=<사례\s*\d+>|$)',
        re.DOTALL
    )
    for m in case_pat.finditer(text):
        num, title, body = m.group(1), m.group(2).strip(), m.group(3).strip()

        # Q / A 분리
        qa = re.search(r'Q\s+(.+?)\s+A\s+(.+)', body, re.DOTALL)
        if qa:
            q = re.sub(r'\s+', ' ', qa.group(1)).strip()
            a = re.sub(r'\s+', ' ', qa.group(2)).strip()
        else:
            q, a = title, re.sub(r'\s+', ' ', body).strip()

        유형 = get_type(m.start())
        chunks.append({
            "chunk_id": f"사례_{num.zfill(2)}",
            "유형번호": 유형["유형번호"],
            "유형명": 유형["유형명"],
            "사례번호": num,
            "제목": title,
            "질문": q,
            "답변": a,
            "content": f"[사례{num}] {title}\nQ: {q}\nA: {a[:600]}",
            "출처": "인공지능기본법 지원데스크 사례집",
            "청크유형": "QA사례"
        })
    print(f"Q&A 사례: {len(chunks)}개 추출")
    return chunks

qa_chunks = extract_qa_cases(part3_text)
if qa_chunks:
    s = qa_chunks[0]
    print(f"\n[사례{s['사례번호']}] {s['제목']}")
    print(f"Q: {s['질문'][:80]}...")
    print(f"A: {s['답변'][:120]}...")


Q&A 사례: 20개 추출

[사례1] 타사의 인공지능으로 제작한 콘텐츠를 사용하면 인공지능이용사업
Q: 타사의 인공지능으로 제작한 콘텐츠를 사용하면 인공지능이용사업...
A: 자에 해당하는가? 23...


## 조항 설명 추출

In [11]:
def extract_articles(text):
    chunks = []
    pat = re.compile(
        r'\d+\.\s*(제[\d·]+조[^|\n]*)\|\s*([^\n]+)\n(.*?)(?=\d+\.\s*제|$)',
        re.DOTALL
    )
    for i, m in enumerate(pat.finditer(text)):
        body = re.sub(r'\s+', ' ', m.group(3)).strip()
        chunks.append({
            "chunk_id": f"조항_{i+1:02d}",
            "조항": m.group(1).strip(),
            "제목": m.group(2).strip(),
            "본문": body[:1500],
            "content": f"{m.group(1).strip()} | {m.group(2).strip()}\n{body[:800]}",
            "출처": "인공지능기본법 지원데스크 사례집",
            "청크유형": "조항설명"
        })
    print(f"조항 설명: {len(chunks)}개 추출")
    return chunks

article_chunks = extract_articles(part2_text)


조항 설명: 6개 추출


## 저장

In [12]:
all_chunks = qa_chunks + article_chunks
print(f"총 청크: {len(all_chunks)}개 (Q&A {len(qa_chunks)} + 조항 {len(article_chunks)})")

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for c in all_chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

print(f"저장 완료 → {OUTPUT_PATH}")


총 청크: 26개 (Q&A 20 + 조항 6)
저장 완료 → 사례집_chunks.jsonl
